In [ ]:
import pandas as pd
import numpy as np

df_contracts = pd.read_pickle("data/contracts_competition.pkl")
df_hockey_scouting = pd.read_csv("data/hockey_scouting_notes.csv")
df_identity_card_0 = pd.read_csv("data/identity_card_0.tsv", sep="\t")
df_identity_card_1 = pd.read_csv("data/identity_card_1.csv")
df_medical_information = pd.read_excel("data/medical_information.xlsx", header=1)
df_moms_notes = pd.read_json("data/moms_notes.json")
df_performance = pd.read_csv("data/performance.tsv", sep="\t")

### id_cards: df_identity_card_0 in latin/roman
the column names and the international_id is in latin/roman:<br>
 - numerus_internationalis_ad_identitatem, numerus_identificationis_ad_medicinam, ...
 - id examples: MMMMMMMDCCVIII, CCLXXV, MMMCMLXV, ...
the thousands ('M') need to be handled separately, they don't conform to the standard notation (too many)

**id_card_0** and **id_card_1** contain exclusive entries, no overlap, and their combination (concat) are exactly the 10.000 desired unique international_id entries.

In [ ]:
import roman

def convert_roman(s: str) -> int:
    rest = s.lstrip("M")  # remove 'M's at the start, because MMMMMMXII is technically illegal and will cause error
    thousands = len(s) - len(rest)  # count how many 'M's were removed
    return roman.fromRoman(rest) + thousands * 1000 if rest else thousands * 1000

def convert_latin_name(s: str, suffix: str) -> str:
    return s.removesuffix(suffix) if s not in ['Marcus', 'Julius'] else s

df_identity_card_0 = df_identity_card_0.rename(
    columns={
        "numerus_internationalis_ad_identitatem": "international_id",
        "numerus_identificationis_ad_medicinam": "medical_id",
        "praenomen": "first_name",
        "cognomen": "last_name",
        "sexus": "gender",
        "aetas_annorum": "age",
        "urbs_natalis": "birth_city",
        "natio": "nationality",
    }
)

df_identity_card_0['international_id'] = df_identity_card_0['international_id'].apply(convert_roman)
df_identity_card_0['first_name'] = df_identity_card_0['first_name'].apply(lambda s: convert_latin_name(s, "us"))
df_identity_card_0['last_name'] = df_identity_card_0['last_name'].apply(lambda s: convert_latin_name(s, "ius"))

df_identity_cards: pd.DataFrame = pd.concat([df_identity_card_0, df_identity_card_1], ignore_index=True)
df_identity_cards['birth_city'] = df_identity_cards['birth_city'].replace({'Kazn': 'Kazan', 'Tmpere': 'Tampere'})  # typos

### df_contracts, df_hockey_scouting, df_identity_cards
regular cleaning: wrong data types, inconsistent categories, typos, ...

In [ ]:
df_contracts['won_championship'] = df_contracts['won_championship'].astype(bool)
df_contracts['captain'] = df_contracts['captain'].astype(bool)

df_hockey_scouting['position'] = df_hockey_scouting['position'].astype('category')
df_hockey_scouting['dominant_hand'] = df_hockey_scouting['dominant_hand'].astype('category')
df_hockey_scouting['experience_level'] = df_hockey_scouting['experience_level'].astype('category')

df_identity_cards["gender"] = df_identity_card_1["gender"].replace({"m": "male", "f": "female"}).astype('category')
df_identity_cards["birth_city"] = df_identity_cards["birth_city"].str.capitalize()
df_identity_cards["nationality"] = df_identity_cards["nationality"].str.replace(".", "").str.capitalize()  # [U.S.A., usa] -> [Usa, Usa]
df_identity_cards["nationality"] = df_identity_cards["nationality"].str.replace("Norwayy", "Norway")

### df_medical_information
The headers are in the first row, and one header is in the second row<br>
height: sometimes in cm, sometimes in m<br>
weight: sometimes contains "kg"<br>
body_fat_percentage: sometimes contains "%"<br>
fitness_level: sometimes contains "nan" instead of actual NaN/null<br>
return_date: e.g. `01-02-2025`, `2025/01/05`, `2025-01-17` (format "mixed" takes care of this)<br>

In [ ]:
df_medical_information = df_medical_information.drop(0)  # row 2 with header, row 1 was handled during import
df_medical_information = df_medical_information.rename(columns={'Unnamed: 0': 'medical_id'})

# convert all heights to cm
df_medical_information['height'] = df_medical_information['height'].str.replace('cm', '')
is_meters = df_medical_information['height'].str.endswith('m')
df_medical_information['height'] = df_medical_information['height'].str.replace('m', '').astype(float)
df_medical_information.loc[is_meters, 'height'] = df_medical_information.loc[is_meters, 'height'] * 100

df_medical_information['weight'] = df_medical_information['weight'].str.replace('kg', '').astype(float)

df_medical_information['body_fat_percentage'] = df_medical_information['body_fat_percentage'].str.replace('%', '').astype(float)

df_medical_information['fitness_level'] = df_medical_information['fitness_level'].str.capitalize()
df_medical_information['fitness_level'] = df_medical_information['fitness_level'].replace('Nan', pd.NA)
df_medical_information['fitness_level'] = df_medical_information['fitness_level'].astype('category')

df_medical_information['has_medical_note'] = df_medical_information['medical_information'].notna()
df_medical_information['return_date'] = pd.to_datetime(df_medical_information['return_date'], errors='raise', format='mixed')
df_medical_information = df_medical_information.drop(columns=['physician_signature'])  # only funny values
df_medical_information = df_medical_information.drop(columns=["age_in_years"])  # exact match to 'age' in other df

# some rows have all NaN, including height. these values also contain nan%, nankg, nancm, Nan, actual NaN. they only have signature, which is considered insignificant
df_medical_information = df_medical_information.dropna(subset=['height'], how='all')

### df_moms_notes
contains many rows that are missing all (significant) values<br>
columns_to_drop do not contain any distinct information. other columns are questionable, but may be useful later.<br>
there is one duplicate for key `['first_name', 'last_name', 'birth_city', 'age']`, duplicate just gets deleted, will not have big impact.<br>
  -> alternative would be having two rows for all combinations, which would lead to international_id not being unique

In [ ]:
df_moms_notes["birth_city"] = df_moms_notes["birth_city"].str.capitalize()
df_moms_notes["school_grade"] = df_moms_notes["school_grade"].round(2)  # remove weird 3.1800000000002 values

df_moms_notes = df_moms_notes.drop(columns=['favourite_child_info', 'favourite_tv_show', 'favourite_food']) # drop columns that are 100% irrelevant
df_moms_notes = df_moms_notes.dropna(subset=['first_name', 'last_name', 'age', 'birth_city'], how='all')
df_moms_notes = df_moms_notes.drop_duplicates(subset=['first_name', 'last_name', 'birth_city', 'age'], keep='first')  # drop second "Lillian"
df_moms_notes['years_in_usa'] = df_moms_notes['years_in_usa'].astype('Int64')

### float -> int
Many columns that should be int are float after import. To not have to handle each column separately, this code block automatically detects which float columns contain only whole numbers, and converts those to int.

'Int64' is chosen instead of int, because it can handle NaN values

In [ ]:
# trying to convert float -> int
def check_if_int(series: pd.Series) -> bool:
    return (series.dropna() % 1 == 0).all()


for df in [df_contracts, df_hockey_scouting, df_identity_cards, df_medical_information, df_moms_notes, df_performance]:
    float_cols = df.select_dtypes(include="float64").columns

    for col in float_cols:
        if check_if_int(df[col]):
            df[col] = df[col].astype("Int64")
            # print(f"converted {col}")

---
### merge analysis
This code produces the table on slide 6. It has no other purpose.

In [ ]:
def merge_overlap_report(left, right, keys, left_name, right_name):
    keys = [keys] if isinstance(keys, str) else keys
    left_keys, right_keys = left[keys].drop_duplicates(), right[keys].drop_duplicates()
    counts = left_keys.merge(right_keys, on=keys, how="outer", indicator=True)["_merge"].value_counts()
    both, left_only, right_only = [counts.get(x, 0) for x in ["both", "left_only", "right_only"]]
    return {
        "left_df": left_name,
        "right_df": right_name,
        "join_keys": ", ".join(keys),
        "left_rows": len(left),
        "right_rows": len(right),
        "left_unique_keys": len(left_keys),
        "right_unique_keys": len(right_keys),
        "matched_keys": both,
        "left_only_keys": left_only,
        "right_only_keys": right_only,
    }

merge_steps = [
    ("contracts", df_hockey_scouting, "international_id", "hockey_scouting", "outer"),
    ("contracts + scouting", df_identity_cards, "international_id", "identity_cards", "outer"),
    ("previous merge", df_medical_information, "medical_id", "medical_information", "left"),
    ("previous merge", df_performance, "international_id", "performance", "outer"),
    ("previous merge", df_moms_notes, ["first_name", "last_name", "birth_city", "age"], "moms_notes", "left"),
]

prev_merge, merge_reports = df_contracts, []
for left_name, right, keys, right_name, how in merge_steps:
    merge_reports.append(merge_overlap_report(prev_merge, right, keys, left_name, right_name))
    prev_merge = prev_merge.merge(right, on=keys, how=how)

merge_report_df = pd.DataFrame(merge_reports)  # merge statistic table of slide 6

### merging datasets
After merging the datasets, all 10.000 distinct international_id entries are still present.<br>
37 rows only have is_captain and won_championship values, but they will be kept.<br>
df_moms_notes is merged with a **left join**, because we want to keep all data where no mom_note can be found, but we don't want to keep mom_notes that cannot be assigned to any international_id.

In [ ]:
# merging the data sets
df_total: pd.DataFrame = (
    df_contracts.merge(df_hockey_scouting, on="international_id", how="outer")
    .merge(df_identity_cards, on="international_id", how="outer")
    .merge(df_medical_information, on="medical_id", how="left")
    .merge(df_performance, on="international_id", how="outer")
    .merge(df_moms_notes, on=['first_name', 'last_name', 'birth_city', 'age'], how="left")
)

df_total_copy = df_total.copy()  # for outlier graphs in Orange.

---
### Outliers
All of these outliers are clearly impossible, and have a (possibly intended) big gap to the realistic values. Graphs of this are in the presentation and in Orange.

Number of outliers that are set to NaN:
 - draft_year < 1950: 40<br>
 - time_on_ice > 100: 2 (regular play goes 60 minutes, outliers are ~500)<br>
 - age > 200: 118<br>
 - height > 250: 50

In [ ]:
# set impossible values to NaN
df_total.loc[df_total['age'] > 200, 'age'] = pd.NA
df_total.loc[df_total["draft_year"] < 1950, "draft_year"] = pd.NA
df_total.loc[df_total["time_on_ice"] > 100, "time_on_ice"] = pd.NA
df_total.loc[df_total['height'] > 250, 'height'] = pd.NA

---
### Contradictions
There are some contradictions in the data, mainly for similar features that have some relationship.
Since we do not know which of two values is correct, we will keep both, and may later take more steps during model construction.
Since it was requested on Discord, but there is not (yet) really something to do, we will for now just count the conflicts.
It can be argued that these assumed relations do not hold if the data is not from the same time, e.g. age [surveyed 2022] != age [surveyed 2023].

**--- honorable mentions ---**
- `shooting_percentage` seems to be random. The values are not even close to the accurate (calculated) values. Thats why we created a new calculated `shooting_percentage_calculated`.
- `num_of_shots` is not the sum of `high_danger_shots`, `medium_danger_shots` and `low_danger_shots`. This would be fine if not all shots are categorized, but 8153 times the sum of categorized shots is higher than the total number of shots, which would only make sense if shots are sometimes categorized in more than one category, which seems unlikely. Also, sometimes one category alone is bigger than the total num_of_shots, which occours for all 3 categories.
- `power_play_goals` is higher than `goals` 8630 times.

The full list of hard contradictions is in the presentation. There are a total of 43878 contradiction instances in 14 different contradiction types.

In [ ]:
# this set made the code much more readable
possible_contradictions = {
  # age vs age_in_years was already checked previously, it was an exact match
  "goals > num_of_shots": df_total["goals"] > df_total["num_of_shots"],
  "goals > shot_attempts": df_total["goals"] > df_total["shot_attempts"],
  "num_of_shots > shot_attempts": df_total["num_of_shots"] > df_total["shot_attempts"],
  "goals > 0 and num_of_shots == 0": (df_total["goals"] > 0) & (df_total["num_of_shots"] == 0),
  "goals > 0 and shot_attempts == 0": (df_total["goals"] > 0) & (df_total["shot_attempts"] == 0),
  "num_of_shots > 0 and shot_attempts == 0": (df_total["num_of_shots"] > 0) & (df_total["shot_attempts"] == 0),

  "goals == 0 and shooting_percentage > 0 and shot_attempts != 0": (df_total["goals"] == 0) & (df_total["shooting_percentage"] > 0) & (df_total["shot_attempts"] != 0),
  "goals > 0 and shooting_percentage == 0": (df_total["goals"] > 0) & (df_total["shooting_percentage"] == 0),
  "shooting_percentage !~ 100 * goals / num_of_shots": (df_total["num_of_shots"] > 0) & ~np.isclose(df_total["shooting_percentage"], 100 * df_total["goals"] / df_total["num_of_shots"], atol=1), # absolute tolerance of 1 (1%) due to rounding difference

  "shot_speed > 0 and num_of_shots == 0": (df_total["shot_speed"] > 0) & (df_total["num_of_shots"] == 0),
  "num_of_shots > 0 and shot_speed == 0": (df_total["num_of_shots"] > 0) & (df_total["shot_speed"] == 0),

  "high + medium + low danger shots > num_of_shots": df_total[["high_danger_shots", "medium_danger_shots", "low_danger_shots"]].sum(axis=1) > df_total["num_of_shots"], # assumes not all shots are categorized as high/med/low danger
  # each feature tested alone in case only one is conflicting
  "high_danger_shots > num_of_shots": df_total["high_danger_shots"] > df_total["num_of_shots"],
  "medium_danger_shots > num_of_shots": df_total["medium_danger_shots"] > df_total["num_of_shots"],
  "low_danger_shots > num_of_shots": df_total["low_danger_shots"] > df_total["num_of_shots"],
  "high_danger_shots > 0 and num_of_shots == 0": (df_total["high_danger_shots"] > 0) & (df_total["num_of_shots"] == 0),
  "medium_danger_shots > 0 and num_of_shots == 0": (df_total["medium_danger_shots"] > 0) & (df_total["num_of_shots"] == 0),
  "low_danger_shots > 0 and num_of_shots == 0": (df_total["low_danger_shots"] > 0) & (df_total["num_of_shots"] == 0),

  "winning_goals > goals": df_total["winning_goals"] > df_total["goals"],
  "winning_goals > 0 and goals == 0": (df_total["winning_goals"] > 0) & (df_total["goals"] == 0),

  "power_play_goals > goals": df_total["power_play_goals"] > df_total["goals"],
  "power_play_goals > 0 and goals == 0": (df_total["power_play_goals"] > 0) & (df_total["goals"] == 0),
  "power_play_goals > 0 and power_play_time == 0": (df_total["power_play_goals"] > 0) & (df_total["power_play_time"] == 0),
  "power_play_time > time_on_ice": df_total["power_play_time"] > df_total["time_on_ice"],

  "puck_recoveries > puck_touches": df_total["puck_recoveries"] > df_total["puck_touches"],
  "puck_recoveries > 0 and puck_touches == 0": (df_total["puck_recoveries"] > 0) & (df_total["puck_touches"] == 0),
  "puck_possession_time > 0 and puck_touches == 0": (df_total["puck_possession_time"] > 0) & (df_total["puck_touches"] == 0),
  "penalty_kill_time > time_on_ice": df_total["penalty_kill_time"] > df_total["time_on_ice"],
  "power_play_time + penalty_kill_time > time_on_ice": df_total["power_play_time"] + df_total["penalty_kill_time"] > df_total["time_on_ice"],
  "puck_possession_time > time_on_ice": df_total["puck_possession_time"] > df_total["time_on_ice"],

  "penalties_taken > 0 and penality_minutes == 0": (df_total["penalties_taken"] > 0) & (df_total["penality_minutes"] == 0),  # assumption: no penalty <1min
  "penality_minutes > 0 and penalties_taken == 0": (df_total["penality_minutes"] > 0) & (df_total["penalties_taken"] == 0),

  "passes_completed > passes_attempted": df_total["passes_completed"] > df_total["passes_attempted"],
  "passes_completed > 0 and passes_attempted == 0": (df_total["passes_completed"] > 0) & (df_total["passes_attempted"] == 0),
  "passes_attempted > 0 and puck_touches == 0": (df_total["passes_attempted"] > 0) & (df_total["puck_touches"] == 0),
  "passes_completed > 0 and puck_touches == 0": (df_total["passes_completed"] > 0) & (df_total["puck_touches"] == 0),
  "passes_completed == 0 and pass_completion_rate > 0 and passes_attempted > 0": (df_total["passes_completed"] == 0) & (df_total["pass_completion_rate"] > 0) & (df_total["passes_attempted"] > 0),
  "passes_attempted > 0 and pass_completion_rate != 100 * passes_completed / passes_attempted": (df_total["passes_attempted"] > 0) & ~np.isclose(df_total["pass_completion_rate"], 100 * df_total["passes_completed"] / df_total["passes_attempted"], atol=1),  # absolute tolerance of 1 (1%) due to rounding difference

  "years_pro > years_played": df_total["years_pro"] > df_total["years_played"],
  "years_played > age": df_total["years_played"] > df_total["age"],
  "years_pro > age": df_total["years_pro"] > df_total["age"],
  "years_in_usa > age": df_total["years_in_usa"] > df_total["age"],
}

total = 0
num_contradicting_features = 0
for description, condition in possible_contradictions.items():
    count = condition.fillna(False).sum()
    if count > 0:
      num_contradicting_features += 1
      total += count
      print(f"{description}: {count}")

print(f"\nTOTAL HARD CONTRADICTIONS: {total}")
print(f"NUMBER OF CONTRADICTING CONDITIONS: {num_contradicting_features}")

################################################################################################################

# new column to compare given values to expected values
df_total["shooting_percentage_calculated"] = (100 * df_total["goals"] / df_total["num_of_shots"]).astype(float)

---
### Correlations
 - Pearson (linear)
 - Shearman (monotonic)
 - Mutual Information (general / non-linear)
 - Grouped Pearson/Shearman (in case correlation is only valid for e.g. Captains or Goalies)

In [ ]:
if False:  # change to True to execute correlation
    from sklearn.feature_selection import mutual_info_regression
    from itertools import combinations

    # Pearson, Shearman
    def matrix_to_pair_list(corr_matrix):
        # keep "upper triangle" without diagonal, since each value is duplicated
        corr_top = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

        corr_pairs = corr_top.stack().reset_index()
        corr_pairs.columns = ['feature1', 'feature2', 'correlation']

        corr_pairs['correlation_abs'] = corr_pairs['correlation'].abs()
        corr_pairs = corr_pairs[corr_pairs['correlation_abs'] > 0.3].dropna()
        corr_pairs = corr_pairs.sort_values(by='correlation_abs', ascending=False)

        return corr_pairs

    numeric_cols = df_total.select_dtypes(include=["number", "Int64", "float64"]).columns

    # Pearson (linear)
    corr_p_matrix = df_total[numeric_cols].corr(method='pearson')
    corr_p_pairs = matrix_to_pair_list(corr_p_matrix)

    # Spearman (monotonic)
    corr_s = df_total[numeric_cols].corr(method='spearman')
    corr_s_pairs = matrix_to_pair_list(corr_s)

    # merge to one list
    correlations = corr_p_pairs.merge(corr_s_pairs, how='outer', on=['feature1','feature2'], suffixes=['_p','_s'])

    #################################################################################

    # mutual info (https://medium.com/@ajitp82/unlocking-hidden-insights-in-exploratory-data-analysis-2b67b84db6e5)
    # only found new [passes_attempted, games_missed_due_to_injury] correlation which does not look interesting
    mi_pairs = []

    for feature1, feature2 in combinations(numeric_cols, 2):
        tmp = df_total[[feature1, feature2]].dropna()
        mi = mutual_info_regression(tmp[[feature1]], tmp[feature2])[0]
        mi_pairs.append([feature1, feature2, mi])

    mi_pairs = pd.DataFrame(mi_pairs, columns=["feature1", "feature2", "mutual_info"])
    mi_pairs = mi_pairs[mi_pairs["mutual_info"] > 0.1]
    correlations = correlations.merge(mi_pairs, how="outer", on=["feature1", "feature2"])

    #############################################################################################

    # grouped correlations (https://medium.com/@ajitp82/unlocking-hidden-insights-in-exploratory-data-analysis-2b67b84db6e5)
    group_cols = ['captain', 'won_championship', 'position', 'dominant_hand', 'experience_level', 'gender', 'nationality', 'fitness_level']
    # found correlations don't look interesting. Graphs found in graphs/correlation/grouped/

    grouped_correlations = []
    for group_col in group_cols:
        for group_value, group_df in df_total.groupby(group_col, dropna=False):
            corr_p = group_df[numeric_cols].corr(method="pearson")
            corr_s = group_df[numeric_cols].corr(method="spearman")

            tmp_p = matrix_to_pair_list(corr_p).rename(columns={"correlation": "correlation_p", "correlation_abs": "correlation_abs_p"})
            tmp_s = matrix_to_pair_list(corr_s).rename(columns={"correlation": "correlation_s", "correlation_abs": "correlation_abs_s"})

            tmp = tmp_p.merge(tmp_s, how="outer", on=["feature1", "feature2"])
            tmp["group_col"] = group_col
            tmp["group_value"] = group_value
            tmp["group_size"] = len(group_df)
            grouped_correlations.append(tmp)

    grouped_correlations = pd.concat(grouped_correlations).reset_index(drop=True)
    grouped_correlations["max_corr"] = grouped_correlations[["correlation_abs_p", "correlation_abs_s"]].max(axis=1)
    grouped_correlations = grouped_correlations.sort_values("max_corr", ascending=False).reset_index(drop=True)

### saving the dataset
The processed dataset is stored in a pickle file, so the part-2 code can be run instantly.<br>
The data is also stored in a csv file, to be accessed in Orange.

In [ ]:
# Save to csv (for orange)
df_total.to_csv("orange/csv/total.csv", index=False)
df_total_copy.to_csv("orange/csv/total_copy.csv", index=False)

# Save to pickle (for python part 2)
df_total.to_pickle("data/processed/data.pkl")